# Immortalite — Self-Play Training (Colab)

Trains the lightweight self-play chess engine on a free GPU. Checkpoints are written to Google Drive so the run survives disconnects.

**Steps:** set runtime to GPU (Runtime -> Change runtime type -> GPU), then run the cells top to bottom.

In [ ]:
# 1. Get the code. Safe to re-run (always works from /content, never nests).
import os
%cd /content
if not os.path.exists('/content/immortalite'):
    !git clone https://github.com/carlo-wong/immortalite.git
%cd /content/immortalite
!git pull --quiet
print('working dir:', os.getcwd())

In [ ]:
# 2. Install dependencies (Colab already has a CUDA torch build).
!pip install -q python-chess numpy tqdm

In [ ]:
# 3. Mount Google Drive for crash-proof checkpoints.
from google.colab import drive
drive.mount('/content/drive')
# Use a fresh directory for clean restart experiments.
CKPT_DIR = '/content/drive/MyDrive/immortalite_checkpoints_v2'
import os; os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints ->', CKPT_DIR)

In [ ]:
# 4. Confirm GPU is available.
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 5. Train (simple baseline config).
# - Uses direct flags instead of C1/C2 staged controls.
# - Resignation can be enabled later once value_sign_acc is reliably > 0.6.
import os, torch

resume = os.path.join(CKPT_DIR, 'latest.pt')
resume_arg = f'--resume {resume}' if os.path.exists(resume) else ''

if torch.cuda.is_available():
    print('GPU detected -> --device cuda --gpu')
    preset = '--device cuda --gpu'
else:
    print('No GPU (quota exhausted?) -> --device cpu --light. Training will be slow.')
    preset = '--device cpu --light'

# Optional resignation (enable later once value head is reliable).
ENABLE_RESIGN = False
RESIGN_THRESHOLD = -0.90
RESIGN_PLIES = 3
RESIGN_MIN_MOVES = 20

# Staged tuning (change ONE at a time, watch metrics each time):
#  A) sims:   add --sims 64               -> watch winrate_vs_prev and seconds
#  B) buffer: add --replay-buffer 120000  -> watch policy_entropy / value_sign_acc
#  C) batch:  add --batch-size 256        -> watch value_loss / grad_norm

resign_arg = ''
if ENABLE_RESIGN:
    resign_arg = (
        f'--resign-threshold {RESIGN_THRESHOLD} '
        f'--resign-plies {RESIGN_PLIES} '
        f'--resign-min-moves {RESIGN_MIN_MOVES}'
    )

!python -m engine.train --iterations 50 {preset} --games 24 --train-steps 150 --concurrency 64 --sims 100 {resign_arg} --save-every 10 --gate-every 10 --gate-games 20 --checkpoint-dir "$CKPT_DIR" $resume_arg

In [ ]:
# 6. Independent Strength Gating (Checkpoints A vs B)
# Compare any two checkpoints (e.g., 'latest' or iteration numbers like 40, 50).
# This runs independently of the training loop and survives disconnects.
import os
import torch
import numpy as np
from engine.config import Config
from engine.network import ChessNet
from engine.train import play_match

# ---- Gating Match Configuration ----
CHECKPOINT_A = 'latest'  # Can be 'latest' or an iteration number (e.g., 50)
CHECKPOINT_B = 40        # Can be 'latest' or an iteration number (e.g., 40)
GAMES = 20               # Number of games to play in the match
SIMS = 100               # Simulations per move
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# ------------------------------------

cfg = Config()

# Resolve Path A
if CHECKPOINT_A == 'latest':
    path_a = os.path.join(CKPT_DIR, 'latest.pt')
    label_a = "Latest"
else:
    path_a = os.path.join(CKPT_DIR, f'ckpt_iter_{CHECKPOINT_A:04d}.pt')
    label_a = f"Iter {CHECKPOINT_A}"

# Resolve Path B
if CHECKPOINT_B == 'latest':
    path_b = os.path.join(CKPT_DIR, 'latest.pt')
    label_b = "Latest"
else:
    path_b = os.path.join(CKPT_DIR, f'ckpt_iter_{CHECKPOINT_B:04d}.pt')
    label_b = f"Iter {CHECKPOINT_B}"

if not os.path.exists(path_a) or not os.path.exists(path_b):
    print(f"Error: Make sure both checkpoints exist!\nPath A: {path_a}\nPath B: {path_b}")
else:
    print(f"Loading Checkpoint A ({label_a}) from {path_a}...")
    state_a = torch.load(path_a, map_location=DEVICE)
    from engine.config import NetConfig
    net_cfg_a = cfg.net
    if isinstance(state_a, dict) and 'net' in state_a:
        net_cfg_a = NetConfig(**state_a['net'])
    net_a = ChessNet(net_cfg_a).to(DEVICE)
    net_a.load_state_dict(state_a['model'] if isinstance(state_a, dict) and 'model' in state_a else state_a, strict=False)
    net_a.eval()

    print(f"Loading Checkpoint B ({label_b}) from {path_b}...")
    state_b = torch.load(path_b, map_location=DEVICE)
    net_cfg_b = cfg.net
    if isinstance(state_b, dict) and 'net' in state_b:
        net_cfg_b = NetConfig(**state_b['net'])
    net_b = ChessNet(net_cfg_b).to(DEVICE)
    net_b.load_state_dict(state_b['model'] if isinstance(state_b, dict) and 'model' in state_b else state_b, strict=False)
    net_b.eval()

    print(f"\nStarting Match: {label_a} vs {label_b} ({GAMES} games, {SIMS} sims, on {DEVICE})...")
    
    metrics = play_match(net_a, net_b, cfg, n_games=GAMES, sims=SIMS, device=DEVICE)
    winrate = metrics['winrate']
    wins = metrics['wins_as_white'] + metrics['wins_as_black']
    losses = metrics['losses_as_white'] + metrics['losses_as_black']
    draws = metrics['draws_as_white'] + metrics['draws_as_black']

    wdl_notation = f"+{wins} ={draws} -{losses}"
    
    print("\n" + "="*40)
    print(f"MATCH COMPLETED!")
    print(f"{label_a} score vs {label_b}: {winrate:.3f} [{wdl_notation}]")
    print(f"  As White: Wins: {metrics['wins_as_white']}, Losses: {metrics['losses_as_white']}, Draws: {metrics['draws_as_white']}")
    print(f"  As Black: Wins: {metrics['wins_as_black']}, Losses: {metrics['losses_as_black']}, Draws: {metrics['draws_as_black']}")
    print(f"  Mean Game Length: {metrics['mean_game_len']:.1f} plies")
    print(f"  Terminations: {metrics['terminations']}")
    if winrate > 0.55:
        print(f"Result: {label_a} PASSED the gate (Stronger than {label_b})!")
    elif winrate < 0.45:
        print(f"Result: {label_a} is significantly WEAKER than {label_b}!")
    else:
        print(f"Result: {label_a} and {label_b} are roughly EQUAL (within noise margin).")
    print("="*40)


In [ ]:
# 7. Plot training progress (run AFTER at least one training iteration finishes).
import os
import pandas as pd
import matplotlib.pyplot as plt

path = os.path.join(CKPT_DIR, 'metrics.csv')
if not os.path.exists(path):
    print('No metrics.csv yet. Wait for iteration 0 of cell 5 to finish '
          '(a few minutes on the --gpu preset), then re-run this cell.')
else:
    df = pd.read_csv(path)
    if df.empty:
        print('metrics.csv exists but is empty — wait for an iteration to finish.')
    else:
        df['iter'] = pd.to_numeric(df['iter'], errors='coerce')
        df = df.dropna(subset=['iter']).copy()
        if df.empty:
            print('metrics.csv has no valid iteration rows yet.')
        else:
            for col in df.columns:
                if col != 'terminations':
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            if {'train_steps', 'batch_size', 'samples'}.issubset(df.columns):
                df['reuse_ratio'] = (
                    df['train_steps'] * df['batch_size'] /
                    df['samples'].replace(0, float('nan'))
                )

            # Gate results live in a separate file (logged independently of
            # metrics.csv so a gate crash never loses per-iteration metrics).
            gates_path = os.path.join(CKPT_DIR, 'metrics_gates.csv')
            gates_df = pd.read_csv(gates_path) if os.path.exists(gates_path) else None
            if gates_df is not None and not gates_df.empty:
                gates_df['iter'] = pd.to_numeric(gates_df['iter'], errors='coerce')
                gates_df['winrate'] = pd.to_numeric(gates_df['winrate'], errors='coerce')
                gates_df = gates_df.dropna(subset=['iter']).copy()

            # Early-run health checks (first 10-20 iterations of a fresh run).
            df_window = df[df['iter'] <= 20].copy()
            if len(df_window) >= 10:
                entropy_ok = float(df_window['policy_entropy'].iloc[-1]) >= 1.8
                sign_acc_series = df_window['value_sign_acc'].dropna()
                value_ok = (not sign_acc_series.empty) and float(sign_acc_series.iloc[-1]) >= 0.6
                gate_rows = int((gates_df['iter'] <= 20).sum()) if gates_df is not None and not gates_df.empty else 0
                gate_ok = gate_rows >= 1
                reuse_series = df_window['reuse_ratio'].dropna() if 'reuse_ratio' in df_window.columns else pd.Series(dtype=float)
                reuse_ok = (not reuse_series.empty) and float(reuse_series.iloc[-1]) <= 8.0
                print('Early-run checks (iter <= 20):')
                print(f"  policy_entropy >= 1.8: {entropy_ok}")
                print(f"  value_sign_acc >= 0.6: {value_ok}")
                print(f"  gate rows present: {gate_ok} (rows={gate_rows})")
                print(f"  reuse_ratio <= 8x: {reuse_ok}")
            else:
                print('Early-run checks need at least 10 completed iterations.')

            fig, ax = plt.subplots(3, 3, figsize=(18, 12))
            ax = ax.flatten()

            def plot_panel(axis, cols, title, ylim=None):
                present = [c for c in cols if c in df.columns]
                if not present:
                    axis.text(0.5, 0.5, 'missing columns', ha='center', va='center')
                    axis.set_title(title)
                    axis.set_xlabel('iteration')
                    return
                df.plot(x='iter', y=present, ax=axis, title=title)
                axis.set_xlabel('iteration')
                if ylim is not None:
                    axis.set_ylim(*ylim)

            plot_panel(ax[0], ['policy_loss', 'value_loss'], 'Losses')
            plot_panel(ax[1], ['policy_entropy', 'grad_norm'], 'Entropy / Gradient norm')
            plot_panel(ax[2], ['value_sign_acc', 'policy_top1_agree'], 'Net target agreement', ylim=(0.0, 1.0))
            plot_panel(ax[3], ['decisive_rate', 'draw_rate', 'max_moves_trunc_rate'], 'Outcome mix', ylim=(0.0, 1.0))
            plot_panel(ax[4], ['mean_game_len'], 'Mean game length')

            if gates_df is None or gates_df.empty:
                ax[5].text(0.5, 0.5, 'no gate rows yet', ha='center', va='center')
                ax[5].set_title('Strength gate (vs prev net)')
                ax[5].set_xlabel('iteration')
            else:
                gates_df.plot(x='iter', y='winrate', marker='o', ax=ax[5], title='Strength gate (vs prev net)')
                ax[5].set_xlabel('iteration')
                ax[5].axhline(0.5, color='gray', linestyle='--', linewidth=1)
                ax[5].set_ylim(0.0, 1.0)

            plot_panel(ax[6], ['learning_rate'], 'Learning rate')
            plot_panel(ax[7], ['reuse_ratio'], 'Reuse ratio (train load / fresh samples)')
            plot_panel(ax[8], ['buffer_size'], 'Replay buffer size')

            plt.tight_layout()
            plt.show()

## After training
Download `latest.pt` from your Drive folder and run the analysis server locally:

```bash
set IMMORTALITE_CHECKPOINT=path\to\latest.pt   # Windows
python -m uvicorn server.app:app --port 8000
```
Then open http://localhost:8000/app/ .